### Convolution
###### 이미지 위에서 stride 값만큼 filter를 이동시키면서 겹쳐지는 부분의 각 원소의 값을 곱해서 모두 더한 값을 출력하는 연산

#### Convolution filter 선언하는 방법
- torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride = 1, padding = 0, bias = True)
- ex) conv = nn.Conv2d(1, 1, 3)  --> 입력 채널 1, 출력 채널 1, 커널 크기 3*3

----

#### 입력의 형태
- input type : torch.Tensor
- input shape : (N * C * H * W) (batch_size, channel, height, width)

----

#### MaxPooling 계층 선언
- torch.nn.MaxPool2d(kernel_size) (나머지는 Default 값이어서 따르 지정 안 해줘도 됨)

----


In [10]:
import torch
import torch.nn as nn

input = torch.Tensor(1, 1, 28, 28)
conv1 = nn.Conv2d(1, 32, 3, padding = 1)
pool1 = nn.MaxPool2d(2)
conv2 = nn.Conv2d(32, 64, 3, padding = 1)
pool2 = nn.MaxPool2d(2)

out1 = conv1(input)
out1.shape
out2 = pool1(out1)
out3 = conv2(out2)
out4 = pool2(out3)
out4.shape

torch.Size([1, 64, 7, 7])

In [18]:
import torch
import torch.nn as nn
import torchvision.datasets as dsets
import torchvision.transforms as transforms
import torch.nn.init

device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed(777)

learning_rate = 0.001
training_epochs = 15
batch_size = 100

mnist_train = dsets.MNIST(root = 'MNIST_data/',
                          train = True,
                          transform = transforms.ToTensor(),
                          download = True)

mnist_test = dsets.MNIST(root = 'MNIST_data/',
                         train = False,
                         transform = transforms.ToTensor(),
                         download = True)

data_loader = torch.utils.data.DataLoader(dataset = mnist_train,
                                          batch_size = batch_size,
                                          shuffle = True,
                                          drop_last = True)

class CNN(nn.Module):

    def __init__(self):

        super(CNN, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Linear(7 * 7 * 64, 10, bias = True)
        torch.nn.init.xavier_normal(self.fc.weight)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)

        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

model = CNN().to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

total_batch = len(data_loader)

for epoch in range(training_epochs):
    avg_cost = 0

    for X, Y in data_loader:
        X = X.to(device)
        Y = Y.to(device)


        optimizer.zero_grad()
        hypothesis = model(X)

        cost = criterion(hypothesis, Y)
        cost.backward()
        optimizer.step()

        avg_cost += cost / total_batch

    print('Epoch: {}, cost: {}'.format(epoch + 1, avg_cost))
print('Learning Finished')


/tmp/ipykernel_116801/1132261751.py:50: FutureWarning: `nn.init.xavier_normal` is now deprecated in favor of `nn.init.xavier_normal_`.
  torch.nn.init.xavier_normal(self.fc.weight)


Epoch: 1, cost: 0.21192894876003265
Epoch: 2, cost: 0.06205856055021286
Epoch: 3, cost: 0.04547073319554329
Epoch: 4, cost: 0.0368540994822979
Epoch: 5, cost: 0.03092174232006073
Epoch: 6, cost: 0.025980595499277115
Epoch: 7, cost: 0.021293049678206444
Epoch: 8, cost: 0.018072735518217087
Epoch: 9, cost: 0.014892508275806904
Epoch: 10, cost: 0.012350048869848251
Epoch: 11, cost: 0.0105635030195117
Epoch: 12, cost: 0.010942036285996437
Epoch: 13, cost: 0.00745984073728323
Epoch: 14, cost: 0.005636499263346195
Epoch: 15, cost: 0.006012630183249712
Learning Finished
